In [1]:
import pandas as pd
from tqdm.notebook import tqdm
import asyncio
import nest_asyncio
import sys
from pathlib import Path

# Add backend directory to path
backend_path = Path("../backend").resolve()
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

from core.rag.search import retrieve_by_vector, retrieve_by_keyword, retrieve_hybrid
from core.rag.llm_factory import get_embeddings
from core.rag.llm_config import LLMConfig

nest_asyncio.apply()
eval_df = pd.read_csv("evaluation_queries.csv")


In [2]:
def get_doc_id(doc):
    metadata = doc.metadata
    if metadata.get("source") == "meeting":
        return str(metadata.get("meeting_id"))
    elif metadata.get("source") == "note":
        return str(metadata.get("id"))
    return ""

def recall_at_k(results, relevant_id, k):
    top_k_ids = [get_doc_id(r) for r in results[:k]]
    return int(str(relevant_id) in top_k_ids)


In [3]:
def reciprocal_rank(results, relevant_id):
    for rank, r in enumerate(results, start=1):
        if get_doc_id(r) == str(relevant_id):
            return 1 / rank
    return 0


In [4]:
async def evaluate_retriever(retriever_fn, name, k_values=[5, 10]):
    recall_scores = {k: [] for k in k_values}
    mrr_scores = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        query = row["query"]
        relevant_id = row["relevant_doc_id"]

        results = await retriever_fn(query, top_k=10)

        # Recall@K
        for k in k_values:
            recall_scores[k].append(
                recall_at_k(results, relevant_id, k)
            )

        # MRR
        mrr_scores.append(
            reciprocal_rank(results, relevant_id)
        )

    # Compute averages
    recall_avg = {k: sum(v)/len(v) for k, v in recall_scores.items()}
    mrr_avg = sum(mrr_scores) / len(mrr_scores)

    print(f"\n===== {name} =====")
    for k in k_values:
        print(f"Recall@{k}: {recall_avg[k]:.4f}")
    print(f"MRR: {mrr_avg:.4f}")

    return recall_avg, mrr_avg


In [5]:
# Set your active project name here
PROJECT_NAME = "Mozilla Issues" # adjust this based on your database

# Setup embeddings for dense & hybrid search
config = LLMConfig(provider="ollama", model="mxbai-embed-large")
embeddings = get_embeddings(config)
embed_query_fn = lambda q: embeddings.embed_query(q)

async def run_evaluations():
    async def bm25_retriever(query, top_k):
        return await retrieve_by_keyword(query, PROJECT_NAME, top_k)

    async def dense_retriever(query, top_k):
        return await retrieve_by_vector(query, PROJECT_NAME, top_k, embed_query_fn)

    async def hybrid_retriever(query, top_k):
        return await retrieve_hybrid(query, PROJECT_NAME, top_k, embed_query_fn)

    await evaluate_retriever(bm25_retriever, "BM25")
    await evaluate_retriever(dense_retriever, "Dense")
    await evaluate_retriever(hybrid_retriever, "Hybrid")

await run_evaluations()


  0%|          | 0/795 [00:00<?, ?it/s]


===== BM25 =====
Recall@5: 0.0088
Recall@10: 0.0088
MRR: 0.0088


  0%|          | 0/795 [00:00<?, ?it/s]


===== Dense =====
Recall@5: 0.4629
Recall@10: 0.5145
MRR: 0.3628


  0%|          | 0/795 [00:00<?, ?it/s]


===== Hybrid =====
Recall@5: 0.4566
Recall@10: 0.5094
MRR: 0.3466
